# L12b Lab: Adaptive Drug Dosing with Q-Learning
Chemotherapy treats solid cancer tumors with drugs that kill fast-dividing cells. While the drug shrinks a tumor, it also damages healthy tissue. This damage, called toxicity, builds up with each dose and clears slowly between doses.

A patient receives chemotherapy in treatment cycles, with one dose in each cycle. A high dose shrinks the tumor faster, but too much toxicity can lead to adverse effects. How much drug should the patient receive in each cycle?

We'll explore this question using the Q-learning update, where we'll learn a dosing policy, and compare the learned policy with the optimal policy.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Model a treatment as a Markov decision process:__ The state of the patient is the size of the tumor and the level of toxicity, and each action is a drug dose. We'll build this model with a reward that penalizes the tumor burden, the toxicity, and the dose.
> * __Compute the optimal dosing policy with value iteration:__ Value iteration uses the equations of the model to compute the best dose in every patient state. We'll compare the learned policy with this optimal policy.
> * __Implement Q-learning to learn a dosing policy:__ A Q-learning agent learns from the rewards and the next states it observes, without the equations of the model. We'll write the Q-learning update, train an agent, and compare the agent's doses with the optimal doses.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths -
include(joinpath(@__DIR__, "Include.jl")); # @__DIR__ locates this notebook's setup file

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to the Julia standard libraries, we use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl) for the Q-learning agent and the policy, [Plots.jl](https://docs.juliaplots.org/stable/) for the figures, and [DataFrames.jl](https://dataframes.juliadata.org/stable/) with [PrettyTables.jl](https://ronisbr.github.io/PrettyTables.jl/stable/) for the tables. See [the package documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) and the [Julia documentation](https://docs.julialang.org/en/v1/) for the functions and types used here.

Your Q-learning function goes in [src/Compute.jl](src/Compute.jl). We call [the `L12bQLearning.myqlearning(...)` function](src/Compute.jl) by its qualified name. This way, re-running the setup cell picks up your edits.
___

## Task 1: Build the dosing environment
In this task, we'll build the dosing environment, which is the Markov decision process that the Q-learning agent acts on.

The state of the patient has two variables. The tumor burden $B\in[0,1]$ is the size of the tumor as a fraction of the largest size the tumor can reach. The toxicity $Z\in[0,1]$ is the damage the drug has built up in the patient. In each treatment cycle $t$, the action is a dose $u_{t}\in\{0,\tfrac{1}{3},\tfrac{2}{3},1\}$, where the four values mean none, low, medium, and high.

The dose changes the state through two equations:

$$
\begin{align*}
B_{t+1} &= B_{t} + \underbrace{g\,B_{t}\,(1 - B_{t})}_{\text{tumor growth}} - \underbrace{k\,u_{t}\,B_{t}}_{\text{tumor killed}}\\
Z_{t+1} &= Z_{t} + \underbrace{b\,u_{t}}_{\text{toxicity added}} - \underbrace{c\,Z_{t}}_{\text{toxicity cleared}}
\end{align*}
$$

where $g$ is the tumor growth rate, $k$ is the fraction of the tumor a full dose kills, $b$ is the toxicity a full dose adds, and $c$ is the fraction of the toxicity the body clears each cycle.

Q-learning stores its values in the Q-table, thus, we need finite state and action spaces. After each cycle, we clip the new values of $B$ and $Z$ to lie between 0 and 1, and we round each value to the nearest of 41 evenly spaced levels. This gives a grid of $41^{2}$ patient states.

The treatment ends when the patient enters an absorbing state. There are two kinds of absorbing states:
* __Cure__: The tumor burden falls to $B\leq B_{\text{cure}}$. The reward is $r_{\text{cure}}>0$.
* __Toxic death__: The toxicity reaches $Z\geq Z_{\text{lethal}}$. The reward is $r_{\text{dead}}<0$. A state that meets both conditions counts as a toxic death.

A cycle that ends in neither absorbing state has a negative reward, a penalty that grows with the tumor burden, the toxicity, and the dose:

$$
\begin{equation*}
r = -\left(w_{B}\,B_{t+1} + w_{Z}\,Z_{t+1} + w_{u}\,u_{t}\right)
\end{equation*}
$$

where $w_{B}$, $w_{Z}$, and $w_{u}$ are nonnegative weights.  We store the environment in [the `MyAdaptiveDosingModel` type](src/Utility.jl), which holds the grid of patient states, the doses, and the parameters. [The `build(...)` function](src/Utility.jl) takes the parameters in a `NamedTuple` and numbers the grid states $1,2,\dots,41^{2}$.

Let's build the environment and save it in the `env::MyAdaptiveDosingModel` variable:

In [ ]:
env = build(MyAdaptiveDosingModel, (
    nlevels = 41, # 41 levels on each axis: B and Z take the values 0, 0.025, ..., 1
    doses = [0.0, 1/3, 2/3, 1.0], # the doses of the four actions: none, low, medium, high
    g = 0.15, # tumor growth rate per cycle
    k = 0.35, # fraction of the tumor a full dose kills per cycle
    b = 0.50, # toxicity a full dose adds per cycle
    c = 0.35, # fraction of the toxicity cleared per cycle
    Bcure = 0.10, # the patient is cured when B ≤ 0.10, 
    Zlethal = 0.90, # the patient dies of toxicity when Z ≥ 0.90
    wB = 1.0, # weight of the tumor burden in the penalty
    wZ = 1.0, # weight of the toxicity in the penalty
    wu = 0.10, # weight of the dose in the penalty
    rcure = 60.0, # reward for a cure
    rdead = -60.0, # reward for a toxic death
));
nstates = env.nlevels^2; # the number of states, 41 × 41
nactions = length(env.doses); # the number of actions, one per dose

We'll follow a treatment that starts from a large tumor and no toxicity, $(B_{0}, Z_{0}) = (0.90, 0)$. [The `stateindex(...)` function](src/Utility.jl) returns the number of the grid state nearest a pair $(B, Z)$. [The `decode(...)` function](src/Utility.jl) does the reverse, and returns the pair $(B, Z)$ of a state number:

In [ ]:
s₀ = stateindex(env, 0.90, 0.0); # the starting state: a large tumor and no toxicity
decode(env, s₀) # check: (B, Z) of the starting state

___

## Task 2: Compute the optimal policy with value iteration
In this task, we'll use value iteration and the equations of the model to compute the optimal policy, the best dose in every patient state.

Each dose leads to exactly one next state $s^{\prime}$. Value iteration updates the value of each state with the Bellman backup:

$$
\begin{equation*}
U(s)\gets\max_{a\in\mathcal{A}}\left(R(s,a) + \gamma\,U(s^{\prime})\right)
\end{equation*}
$$

where the value $U(s)$ is the largest total discounted reward that the treatment can collect starting from state $s$, $\mathcal{A}$ is the set of four doses, $R(s,a)$ is the reward of dose $a$ in state $s$, and $\gamma$ is the discount factor.

[The `solve(...)` function](src/Utility.jl) applies this backup to every state until the values stop changing. The function returns the optimal action-value table $Q^{*}$, where the entry $Q^{*}(s,a)$ is the total discounted reward of giving dose $a$ in state $s$ and then following the optimal policy. [The `policy(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/mdp/#VLDataScienceMachineLearningPackage.policy-Tuple{Matrix{Float64}}) picks the action with the largest $Q^{*}(s,a)$ in each state.

Let's compute the optimal action values and the optimal policy:

In [ ]:
γ = 0.95; # discount factor
Qstar = solve(env; γ = γ); # value iteration: the optimal action-value table
π_star = policy(Qstar); # the optimal policy: the best action in each state

[The `simulate(...)` function](src/Utility.jl) follows a policy from a starting state until the patient is cured, dies of toxicity, or 100 cycles pass. Let's follow the optimal policy from the starting state and print the dose it gives each cycle:

In [ ]:
optimal = simulate(env, π_star, s₀); # follow the optimal policy from the large tumor
println("Outcome: ", optimal.outcome, " after ", optimal.steps, " cycles");
println("Doses: ", round.(optimal.dose, digits = 2));

### Visualize the treatment
Let's draw the tumor burden, the toxicity, and the dose over the treatment cycles:

In [ ]:
let
    cycles = range(0, stop = length(optimal.B) - 1, step = 1) |> collect; # the cycle numbers 0, 1, 2, ...

    # the tumor burden and the toxicity after each cycle -
    plot(cycles, optimal.B, lw = 3, c = :dodgerblue3, marker = :circle, ms = 4, msc = :dodgerblue3, label = "tumor burden B");
    plot!(cycles, optimal.Z, lw = 3, c = :red, marker = :circle, ms = 4, msc = :red, label = "toxicity Z");

    # the dose in each cycle, in the color of the legend text: black on the light theme, white on the dark theme -
    dose_color = current()[1][:legend_font_color];
    plot!(cycles, optimal.dose, seriestype = :steppost, lw = 2, ls = :dash, c = dose_color, label = "dose u");

    # the levels that end the treatment -
    hline!([env.Bcure], lw = 2, ls = :dash, c = :dodgerblue3, label = "cure level");
    hline!([env.Zlethal], lw = 2, ls = :dash, c = :red, label = "lethal level");

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    xlabel!("Treatment cycle");
    ylabel!("Tumor burden, toxicity, and dose");
    plot!(background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
        legend = :outertop, legend_column = 3, xlims = (-0.3, optimal.steps + 0.3), ylims = (-0.03, 1.05), size = (760, 460));
end

The tumor burden falls with each high dose and grows back a little during each rest. The toxicity rises with each high dose, falls during each rest, and peaks at 0.875, just below the lethal level of 0.90.

In [ ]:
# follow the patient after the last dose: the equations of Task 1 with no dose -
number_of_cycles_after = 40; # the number of cycles to follow after the treatment ends
B_after = [optimal.B[end]]; # the tumor burden at the end of the treatment
Z_after = [optimal.Z[end]]; # the toxicity at the end of the treatment
for t ∈ 1:number_of_cycles_after
    B = B_after[end]; # the tumor burden at the start of this cycle
    Z = Z_after[end]; # the toxicity at the start of this cycle
    push!(B_after, B + env.g*B*(1 - B)); # with no dose, the tumor only grows
    push!(Z_after, Z - env.c*Z); # with no dose, the toxicity only clears
end

# draw the treatment and the cycles after it -
let
    # join the two periods; entry 1 of B_after and Z_after repeats the last state of the treatment -
    B_all = vcat(optimal.B, B_after[2:end]);
    Z_all = vcat(optimal.Z, Z_after[2:end]);
    dose_all = vcat(optimal.dose, zeros(number_of_cycles_after)); # no dose after the treatment
    cycles = range(0, stop = length(B_all) - 1, step = 1) |> collect; # the cycle numbers 0, 1, 2, ...

    # the tumor burden and the toxicity -
    plot(cycles, B_all, lw = 3, c = :dodgerblue3, label = "tumor burden B");
    plot!(cycles, Z_all, lw = 3, c = :red, label = "toxicity Z");

    # the dose, in the color of the legend text: black on the light theme, white on the dark theme -
    dose_color = current()[1][:legend_font_color];
    plot!(cycles, dose_all, seriestype = :steppost, lw = 2, ls = :dash, c = dose_color, label = "dose u");

    # the cure level and the last cycle of the treatment -
    hline!([env.Bcure], lw = 2, ls = :dash, c = :dodgerblue3, label = "cure level");
    vline!([optimal.steps], lw = 2, ls = :dash, c = :gray50, label = "end of treatment");

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    xlabel!("Cycle");
    ylabel!("Tumor burden, toxicity, and dose");
    plot!(background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
        legend = :outertop, legend_column = 3, xlims = (0, length(B_all) - 1), ylims = (-0.03, 1.05), size = (760, 460));
end

___

## Task 3: Implement Q-learning and learn a dosing policy
In Task 2, value iteration used the equations of the model. In this task, we suppose the agent does not know those equations. The agent can only give a dose and observe the reward and the next state that [the `world(...)` function](src/Utility.jl) returns.

Q-learning uses these observations to estimate the action-value table $Q$, with one row per state and one column per action. After each dose, the agent updates one entry of the table:

$$
\begin{equation*}
Q(s,a)\gets Q(s,a)+\alpha\left(r+\gamma\max_{a^{\prime}\in\mathcal{A}}Q(s^{\prime},a^{\prime})-Q(s,a)\right)
\end{equation*}
$$

where $0<\alpha\leq 1$ is the learning rate and $\gamma$ is the discount factor. The term $r+\gamma\max_{a^{\prime}}Q(s^{\prime},a^{\prime})$ is the temporal difference target, and its difference from $Q(s,a)$ is the temporal difference error.

The agent repeats this update over many simulated treatments, which we call episodes. Here is the full algorithm:

__Initialize__: Given the state space $\mathcal{S}$, the action space $\mathcal{A}$, the absorbing states, a discount factor $0\leq\gamma<1$, a number of episodes $N\geq 1$, and a maximum number of steps per episode $d\geq 1$, set $Q(s,a)\gets 0$ and the visit count $n(s,a)\gets 0$ for all $s\in\mathcal{S}$ and $a\in\mathcal{A}$.

For $i = 1$ to $N$ __do__:
1. Set the exploration probability $\epsilon$ of episode $i$ from its schedule.
2. Choose a starting state $s$ that is not absorbing, uniformly at random, and set the step counter $t\gets 0$.
3. While $s$ is not absorbing and $t<d$ __do__:
    - Choose the action $a$: roll a random number $p\in[0,1]$.
        - If $p\leq\epsilon$ __then__ choose $a\in\mathcal{A}$ uniformly at random (exploration).
        - Otherwise, choose the greedy action $a\gets\arg\max_{a^{\prime}\in\mathcal{A}}Q(s,a^{\prime})$ (exploitation).
    - Take action $a$, and observe the reward $r$ and the next state $s^{\prime}$ from the world.
    - Count the visit, $n(s,a)\gets n(s,a)+1$, and set the learning rate $\alpha\gets 1/n(s,a)$.
    - Update the action value with the temporal difference error:
      $$Q(s,a)\gets Q(s,a)+\alpha\left(r+\gamma\max_{a^{\prime}\in\mathcal{A}}Q(s^{\prime},a^{\prime})-Q(s,a)\right)$$
    - Update the state $s\gets s^{\prime}$ and the step counter $t\gets t+1$.

__Extract Policy__: For each state $s\in\mathcal{S}$, compute $\pi(s)\gets\arg\max_{a\in\mathcal{A}}Q(s,a)$.

> __Training:__ The agent trains with these settings:
> * Each episode starts from a random state that is not absorbing. The episode runs until the patient is cured, the patient dies, or 80 cycles pass.
> * The learning rate of the $n$-th update of a pair $(s,a)$ is $1/n$. This rate meets the convergence conditions of the [lecture](../L12a/CHEME-5800-L12a-Lecture-QLearning-Fall-2026.ipynb).
> * The exploration probability $\epsilon$ falls in a straight line from 1, reaches 0.05 after 66.5 percent of the episodes, and then stays at 0.05.

We store the agent in a `MyQLearningAgentModel` from the course package, which holds the states, the actions, a learning rate $\alpha$, the discount factor $\gamma$, and the table $Q$. [The `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/) makes the agent.

The agent also needs the states that end a treatment, the cures and the toxic deaths. [The `isterminal(...)` function](src/Utility.jl) returns `true` for these states. Let's build the agent, and collect the absorbing states in the `absorbing_states::Set{Int64}` set:

In [ ]:
# let's build our Q-learning agent, which is a table of action values Q(s,a) for each state-action pair (s,a) -
agent = build(MyQLearningAgentModel, (
    states = collect(1:nstates), # the states 1, 2, ..., nstates
    actions = collect(1:nactions), # the actions 1, 2, 3, 4
    α = 0.1, # learning rate field (myqlearning(...) uses 1/n instead)
    γ = γ, # the same discount factor as value iteration
    Q = zeros(nstates, nactions), # the table starts at zero
));

# These are our absorbing states -
absorbing_states = Set{Int64}(); # the states that end a treatment: cured or dead
for s ∈ 1:nstates
    if (isterminal(env, s) == true)
        push!(absorbing_states, s);
    end
end

### Implement Our Version
Open [`src/Compute.jl`](src/Compute.jl) and complete [the `myqlearning(...)` function](src/Compute.jl). A TODO comment marks each of the three lines you write:

> __Implementation tasks__
>
> * __TODO 1:__ Choose the greedy action in the exploitation branch of the epsilon-greedy rule.
> * __TODO 2:__ Compute the temporal difference target.
> * __TODO 3:__ Move $Q(s,a)$ toward the target with the learning rate $\alpha$.

Our version returns a `NamedTuple` holding the learned table `Q` and the counts `visits`, where `visits[s,a]` is the number of times the agent took action `a` in state `s`. A rerun with the same `seed` gives the same result.

Save the file, re-run the setup cell to reload your edits, then train the agent. The untouched function throws an error.

In [ ]:
result = L12bQLearning.myqlearning(agent, env; worldmodel = world, absorbing = absorbing_states,
    episodes = 200_000, maxsteps = 80, seed = 1234); # the untouched stub throws an error
π_q = policy(result.Q); # the learned policy: the best action in each row of the learned table

### Follow the learned policy
Let's follow the learned policy from the same starting state:

In [ ]:
learned = simulate(env, π_q, s₀); # follow the learned policy from the large tumor
println("Outcome: ", learned.outcome, " after ", learned.steps, " cycles");
println("Doses: ", round.(learned.dose, digits = 2));

Let's also follow both policies from four starting tumor burdens:

In [ ]:
let
    df = DataFrame(); # one row per starting tumor burden
    for B₀ ∈ [0.30, 0.50, 0.70, 0.90]
        s = stateindex(env, B₀, 0.0); # this start, with no toxicity
        a = simulate(env, π_star, s); # the optimal policy
        b = simulate(env, π_q, s); # the learned policy
        push!(df, (B₀ = B₀, optimal = String(a.outcome), optimal_cycles = a.steps,
            learned = String(b.outcome), learned_cycles = b.steps));
    end
    pretty_table(df; backend = :text, display_size = (-1, -1),
        table_format = TextTableFormat(borders = text_table_borders__compact));
end

In [ ]:
let
    
    # initialize --
    B₀_values = [0.30, 0.50, 0.70, 0.90]; # the starting tumor burdens, each with no toxicity
    colors = [:seagreen, :darkorange, :dodgerblue3, :red]; # one color per starting tumor burden
    policies = [π_star, π_q]; # the policy of the left panel and of the middle panel
    titles = ["Optimal policy (value iteration)", "Learned policy (Q-learning)"];
    panels = []; # the two state figures, then one small dose figure per starting tumor burden

    for i ∈ 1:2
        p = plot(title = titles[i], titlefontsize = 12);

        # follow this policy from each starting tumor burden -
        for j ∈ 1:length(B₀_values)
            result = simulate(env, policies[i], stateindex(env, B₀_values[j], 0.0));
            cycles = range(0, stop = length(result.B) - 1, step = 1) |> collect; # the cycle numbers 0, 1, 2, ...
            plot!(p, cycles, result.B, lw = 3, c = colors[j], label = "B₀ = $(B₀_values[j])");
            plot!(p, cycles, result.Z, lw = 2, ls = :dash, c = colors[j], label = ""); # "" leaves this curve out of the legend
        end

        # the cure level and the lethal level -
        hline!(p, [env.Bcure, env.Zlethal], lw = 2, ls = :dot, c = :gray50, label = "cure and lethal levels");

        # labels and style: the gray panel is see-through, so the figure works in both themes -
        xlabel!(p, "Treatment cycle");
        ylabel!(p, "Tumor burden (solid), toxicity (dashed)");
        plot!(p, background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
            legend = :outertop, legend_column = 3, ylims = (-0.03, 1.05));
        push!(panels, p);
    end

    # the learned doses are drawn in the color of the legend text: black on the light theme, white on the dark theme -
    learned_color = current()[1][:legend_font_color];

    # the dose schedules of both policies, one small figure per starting tumor burden -
    for j ∈ 1:length(B₀_values)
        s = stateindex(env, B₀_values[j], 0.0); # this start, with no toxicity
        optimal_run = simulate(env, π_star, s);
        learned_run = simulate(env, π_q, s);
        optimal_cycles = range(0, stop = length(optimal_run.dose) - 1, step = 1) |> collect;
        learned_cycles = range(0, stop = length(learned_run.dose) - 1, step = 1) |> collect;

        # :steppost holds each dose flat until the next cycle -
        q = plot();
        plot!(q, optimal_cycles, optimal_run.dose, seriestype = :steppost, lw = 3, c = colors[j], label = "");
        plot!(q, learned_cycles, learned_run.dose, seriestype = :steppost, lw = 2, ls = :dash, c = learned_color, label = "");

        # only the top figure gets a title, and only the bottom figure gets the cycle numbers and the axis label -
        if (j == 1)
            title!(q, "Doses: optimal (solid), learned (dashed)");
        end
        if (j == length(B₀_values))
            xlabel!(q, "Treatment cycle");
        else
            plot!(q, xformatter = :none); # no cycle numbers under the upper figures
        end
        ylabel!(q, "B₀ = $(B₀_values[j])");
        plot!(q, background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, titlefontsize = 11,
            yticks = ([0, 1/3, 2/3, 1], ["0", "1/3", "2/3", "1"]), ylims = (-0.1, 1.1));
        push!(panels, q);
    end

    # the two state figures, then the four dose figures stacked in a third column, all on the same cycle axis -
    layout = @layout [a{0.35w} b{0.35w} grid(4, 1)]; # each state figure gets 35% of the width, the dose column the rest
    plot(panels..., layout = layout, link = :x, size = (1400, 540), bottom_margin = 5Plots.mm, left_margin = 6Plots.mm);
end

Both policies cure the patient from all four starting tumor burdens. The learned policy takes at most one more cycle than the optimal policy.
___

## Summary
In this lab, we modeled a course of chemotherapy as a Markov decision process, computed the optimal dosing policy with value iteration, and learned a dosing policy with our own Q-learning function.

> __Key Takeaways:__
>
> * __Dosing is a sequential decision problem:__ A larger dose kills more of the tumor and adds more toxicity, and both changes carry into later cycles. The optimal dose changed from cycle to cycle as the tumor burden and the toxicity changed.
>
> * __The optimal dosing schedule:__ Value iteration used the equations of the model to find a schedule of high doses with rest cycles that let the toxicity clear. The schedule kept the toxicity below the lethal level while the tumor shrank to the cure level.
>
> * __Learning the doses from simulated treatments:__ Our Q-learning agent used only observed rewards and next states, and the agent gave the same doses as the optimal policy from the large starting tumor. From the four starting tumor burdens we compared, both policies cured the patient, and the learned policy took at most one more cycle.

We can now change the doses, the rewards, or the number of episodes and see how the optimal policy and the learned policy respond.
___